# VoiceForge -- Day 5: Colab Setup + Smoke-Test Training Run

**Goal for today:** get every moving piece (dataset, base model, QLoRA
config, trainer) wired up and confirm a few training steps run cleanly on
a free T4 GPU with no OOM -- *before* committing to the full run in Day 6.

**Before running:** `Runtime > Change runtime type > T4 GPU`, and add your
Hugging Face token as a Colab secret named `HF_TOKEN`
(key icon in the left sidebar > add secret > enable notebook access). 
Used here direct interactive prompt to mannually enter the hf token.


## 1. Install dependencies

In [1]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 92.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 125.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.6 MB/s eta 0:00:00:00:0100:01


## 2. GPU check

Also detects whether the GPU supports **bf16** natively. This matters more
than it might seem: Colab's free-tier T4 is a Turing-generation GPU
(compute capability 7.5), which does **not** have native bf16 tensor-core
support -- Ampere or newer (capability 8.0+) is required for that. Training
with `bf16=True` on a T4 will run but silently either fall back to a slow
path or produce degraded numerics depending on your stack version. We
detect this and use fp16 automatically on T4, bf16 automatically if you
happen to get an A100/L4 (Colab occasionally offers better GPUs even on
free tier).


In [2]:
import torch

assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > T4 GPU."

print("GPU:", torch.cuda.get_device_name(0))
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {total_vram_gb:.1f} GB")

major, minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Compute capability: {major}.{minor} -> using {'bf16' if USE_BF16 else 'fp16'} "
      f"({'native tensor-core support' if USE_BF16 else 'T4-class GPU, bf16 not natively supported'})")


GPU: Tesla T4
VRAM: 15.6 GB
Compute capability: 7.5 -> using fp16 (T4-class GPU, bf16 not natively supported)


## 3. Log in and load the dataset from the Hub

In [4]:
import os
from huggingface_hub import login

# 1. Try fetching from environment variables (.env file or system shell)
hf_token = os.getenv("HF_TOKEN")

# 2. Only attempt google.colab.userdata if explicitly running in the native Colab web browser UI
# (We check if we are in Colab AND not running via VS Code extension / remote proxy)
if not hf_token:
  try:
    # VS Code Colab extension usually sets specific environment variables or lacks the UI bridge.
    # We can check for standard Colab shell indicators, but safe exception catching is best:
    from google.colab import userdata

    # Only call if not timing out or if you actually registered it in the Colab UI sidebar.
    # If you are in VS Code, skip this and use os.environ instead.
    print(
        "Skipping Colab userdata lookup (running in VS Code extension). Please"
        " set HF_TOKEN as an environment variable."
    )
  except (ImportError, ModuleNotFoundError):
    pass

# Fallback: If token is still missing, prompt interactively or raise a clear error
if not hf_token:
  import getpass

  hf_token = getpass.getpass("Enter your Hugging Face Token (HF_TOKEN): ")

# Authenticate
login(token=hf_token)
print("Successfully authenticated with Hugging Face!")

from datasets import load_dataset

DATASET_REPO = "bikalpoudel/voiceforge-brand-voice-sft"
ds = load_dataset(DATASET_REPO)

print(ds)
print("\nExample brief:     ", ds["train"][0]["brief"])
print("Example completion:", ds["train"][0]["completion"])
print("Example voice:      ", ds["train"][0]["voice"])

Skipping Colab userdata lookup (running in VS Code extension). Please set HF_TOKEN as an environment variable.
Successfully authenticated with Hugging Face!


README.md:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.63MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/194 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/24 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'voice', 'voice_tag', 'format', 'product', 'angle', 'brief', 'completion', 'messages'],
        num_rows: 194
    })
    validation: Dataset({
        features: ['id', 'voice', 'voice_tag', 'format', 'product', 'angle', 'brief', 'completion', 'messages'],
        num_rows: 24
    })
    test: Dataset({
        features: ['id', 'voice', 'voice_tag', 'format', 'product', 'angle', 'brief', 'completion', 'messages'],
        num_rows: 25
    })
})

Example brief:      [Voice: Romantic Floral] Write a restock/drop announcement (15-30 words) for a custom reference order. Campaign angle: care tip or fun fact about the piece. Include a natural CTA encouraging DMs for orders or custom requests. Frame it as 'back' or 'new'.
Example completion: Our custom reference bouquets are back in stock 💐 Never needs water, just a sweet spot on your shelf. Send a reference picture in DM to start your custom order ✨
Example voice:       romantic_flor

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata

# import os
# import getpass

# # prompt interactively to request hf token
# os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")

# # login(token=userdata.get("HF_TOKEN"))
# # Authenticate
# login(token=hf_token)
# print("Successfully authenticated with Hugging Face!")

# from datasets import load_dataset

# DATASET_REPO = "bikalpoudel/voiceforge-brand-voice-sft"
# ds = load_dataset(DATASET_REPO)
# print(ds)
# print()
# print("Example brief:     ", ds["train"][0]["brief"])
# print("Example completion:", ds["train"][0]["completion"])
# print("Example voice:      ", ds["train"][0]["voice"])


## 4. System prompt strategy 

The `messages` field already baked into the dataset uses the **full**
persona section of `voice_guidelines.md` as the system prompt (~2,000
tokens per persona) -- that's the right choice for Day 3's teacher-model
generation, where a strong model needs the whole style guide to nail an
unfamiliar voice zero-shot.

It's the **wrong** choice to repeat verbatim on every SFT training
example. Only the completion tokens (~40-90 of them) carry gradient
signal per example; a ~2,000-token system prompt multiplies the
compute/memory cost of every single training step for no additional
learning signal -- the model is meant to learn the voice from seeing
hundreds of (tag, brief, completion) examples, not from re-reading the
full style guide each time.

Default here is a short **distilled** system prompt per persona (tone
words, banned words, emoji policy, one line on concrete details) instead.
Set `USE_DISTILLED_SYSTEM_PROMPT = False` to fall back to the full
dataset-provided prompt if you'd rather test that trade-off yourself --
try it only if the smoke test below has comfortable memory headroom.


In [5]:
USE_DISTILLED_SYSTEM_PROMPT = True

DISTILLED_SYSTEM_PROMPTS = {
    "cozy_crochet": (
        "You are a copywriter for a cozy handmade crochet shop. Voice: genuine, "
        "cozy, cute, simple, thoughtful. Short sentences, contractions and "
        "fragments are fine. Emoji: choose from \U0001F9F6 \U0001F338 \u2728 \U0001F49B, max 1-2 per post. Never say: "
        "elevate, premium, luxurious, game-changer, exclusive drop. Include one "
        "concrete detail (size, turnaround time, or material) when the format "
        "calls for it. Output only the copy itself, no labels or quotation marks."
    ),
    "romantic_floral": (
        "You are a copywriter for a romantic handmade bouquet shop. Voice: "
        "sentimental, heartfelt, aesthetic, warm, custom-focused. Emoji: choose "
        "from \U0001F490 \U0001F380 \U0001F48C \u2728, max 1-2 per post. Never say: unlock, revolutionary, "
        "unrivaled, cheap, bulk, flash sale. Include one concrete detail (size, "
        "turnaround time, or materials) when the format calls for it. Output "
        "only the copy itself, no labels or quotation marks."
    ),
}

def build_example_text(example, tokenizer):
    system_content = (
        DISTILLED_SYSTEM_PROMPTS[example["voice"]]
        if USE_DISTILLED_SYSTEM_PROMPT
        else example["messages"][0]["content"]
    )
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": example["brief"]},
        {"role": "assistant", "content": example["completion"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


## 5. Load the base model in 4-bit (QLoRA)

Default is **Qwen2.5-7B-Instruct** -- ungated on the Hub, so there's no
license-request wait before your `HF_TOKEN` can download it. Llama-3.1-8B-
Instruct is a fine alternative but requires requesting access on its model
page and getting approved first; swap `BASE_MODEL` once that clears if you
prefer it.


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # requires accepting the license first

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
)
model.config.use_cache = False  # required alongside gradient checkpointing

print(f"Loaded {BASE_MODEL} in 4-bit.")
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct in 4-bit.
Memory allocated: 5.56 GB


## 6. Format the dataset

In [7]:
train_ds = ds["train"].map(
    lambda ex: build_example_text(ex, tokenizer),
    remove_columns=ds["train"].column_names,
)
val_ds = ds["validation"].map(
    lambda ex: build_example_text(ex, tokenizer),
    remove_columns=ds["validation"].column_names,
)

print(f"train: {len(train_ds)} rows, val: {len(val_ds)} rows")
print()
print(train_ds[0]["text"][:600], "...")


Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

train: 194 rows, val: 24 rows

<|im_start|>system
You are a copywriter for a romantic handmade bouquet shop. Voice: sentimental, heartfelt, aesthetic, warm, custom-focused. Emoji: choose from 💐 🎀 💌 ✨, max 1-2 per post. Never say: unlock, revolutionary, unrivaled, cheap, bulk, flash sale. Include one concrete detail (size, turnaround time, or materials) when the format calls for it. Output only the copy itself, no labels or quotation marks.<|im_end|>
<|im_start|>user
[Voice: Romantic Floral] Write a restock/drop announcement (15-30 words) for a custom reference order. Campaign angle: care tip or fun fact about the piece. Inc ...


## 7. Configure QLoRA

In [8]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


## 8. Smoke test: a handful of steps, not a full run

`max_steps=5` here is the whole point of Day 5 -- confirm this runs
cleanly (no OOM, loss looks sane) before Day 6 removes the cap and trains
for real. `max_seq_length=2048` accounts for the system prompt (even the
distilled one, plus the brief and completion) -- raise it only if you see
truncation warnings, since every extra token costs memory on a free GPU.


In [11]:
from trl import SFTTrainer, SFTConfig

smoke_config = SFTConfig(
    output_dir="/content/voiceforge-smoke-test",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=5,                        # Day 6 removes this in favor of num_train_epochs
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    logging_steps=1,
    max_length=2048,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=smoke_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
1,4.599258
2,3.766877
3,3.222723
4,3.017249
5,2.829592


TrainOutput(global_step=5, training_loss=3.4871397495269774, metrics={'train_runtime': 467.9122, 'train_samples_per_second': 0.171, 'train_steps_per_second': 0.011, 'total_flos': 788039835555840.0, 'train_loss': 3.4871397495269774, 'epoch': 0.41237113402061853})

## 9. Check memory headroom and trainable params

In [12]:
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak memory allocated: {peak_gb:.2f} GB / {total_vram_gb:.1f} GB available")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of {total:,} total)")


Peak memory allocated: 10.01 GB / 15.6 GB available
Trainable params: 40,370,176 (0.92% of 4,393,342,464 total)


## Checklist before moving to Day 6

- **Loss is decreased** across the 5 logged steps (not flat, not NaN).
- **Peak memory** comfortably under ~13-14 (10.01) GB  -- leave headroom below the
  T4's 15 GB, since the full run adds eval passes and longer wall-clock
  exposure to fragmentation.
- **Trainable params** should land around 1% of total params for `r=16`
  on a 7-8B model (tens of millions, not hundreds).
- No OOM, no dtype warnings, no truncation warnings.

Next, Day 6 is: remove `max_steps`, set
`num_train_epochs=2`, add `eval_strategy="steps"`, run the full training
loop, then save and push the adapter to the Hub.
